In [ ]:
import dataclasses

import jax

from openpi.models import model as _model
from openpi.policies import droid_policy
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader

In [2]:
download.maybe_download("s3://openpi-assets/checkpoints/pi0_base")
# curl -L -o /usr/local/bin/oss https://gpucloud-static-public-prod.gpushare.com/installation/oss/oss_linux_x86_64
# chmod u+x /usr/local/bin/oss

OH YEAH


  0%|          | 0.00/11.2G [00:00<?, ?iB/s]

OH YEAH


PosixPath('/hy-tmp/likaiyu/resources/openpi-assets/checkpoints/pi0_base')

# Policy inference

The following example shows how to create a policy from a checkpoint and run inference on a dummy example.

In [2]:
config = _config.get_config("pi0_fast_droid")
checkpoint_dir = download.maybe_download("s3://openpi-assets/checkpoints/pi0_fast_droid")

# Create a trained policy.
policy = _policy_config.create_trained_policy(config, checkpoint_dir)

# Run inference on a dummy example. This example corresponds to observations produced by the DROID runtime.
example = droid_policy.make_droid_example()
result = policy.infer(example)

# Delete the policy to free up memory.
del policy

print("Actions shape:", result["actions"].shape)

OH YEAH
OH YEAH


A new version of the following files was downloaded from https://huggingface.co/physical-intelligence/fast:
- processing_action_tokenizer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Some kwargs in processor config are unused and will not have any effect: action_dim, time_horizon, min_token, scale, vocab_size. 


OH YEAH


A new version of the following files was downloaded from https://huggingface.co/physical-intelligence/fast:
- processing_action_tokenizer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Some kwargs in processor config are unused and will not have any effect: action_dim, time_horizon, min_token, scale, vocab_size. 


Actions shape: (10, 8)


# Working with a live model


The following example shows how to create a live model from a checkpoint and compute training loss. First, we are going to demonstrate how to do it with fake data.


In [3]:
config = _config.get_config("pi0_aloha_sim")

checkpoint_dir = download.maybe_download("s3://openpi-assets/checkpoints/pi0_aloha_sim")
key = jax.random.key(0)

# Create a model from the checkpoint.
model = config.model.load(_model.restore_params(checkpoint_dir / "params"))

# We can create fake observations and actions to test the model.
obs, act = config.model.fake_obs(), config.model.fake_act()

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)
print("Loss shape:", loss.shape)

OH YEAH


  0%|          | 0.00/11.2G [00:00<?, ?iB/s]

OH YEAH
Loss shape: (1, 50)


Now, we are going to create a data loader and use a real batch of training data to compute the loss.

In [ ]:
# Reduce the batch size to reduce memory usage.
config = dataclasses.replace(config, batch_size=2)

# Load a single batch of data. This is the same data that will be used during training.
# NOTE: In order to make this example self-contained, we are skipping the normalization step
# since it requires the normalization statistics to be generated using `compute_norm_stats`.
loader = _data_loader.create_data_loader(config, num_batches=1, skip_norm_stats=True)
obs, act = next(iter(loader))

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)

# Delete the model to free up memory.
del model

print("Loss shape:", loss.shape)

In [5]:
loss

Array([[0.16215318, 0.16935264, 0.1718472 , 0.17433359, 0.16571105,
        0.17201906, 0.1543584 , 0.15120131, 0.15917733, 0.15549305,
        0.145278  , 0.14355513, 0.14379016, 0.14728257, 0.15699548,
        0.15274869, 0.15380824, 0.16185635, 0.1658678 , 0.17056672,
        0.17922118, 0.19198453, 0.1866706 , 0.20555466, 0.2204352 ,
        0.22983834, 0.24877393, 0.27449888, 0.27996916, 0.30573726,
        0.33641937, 0.36290833, 0.3624642 , 0.38098663, 0.41549057,
        0.45667052, 0.4873895 , 0.52096546, 0.5432195 , 0.6146357 ,
        0.6257803 , 0.67860043, 0.68782854, 0.7339193 , 0.7973938 ,
        0.79219234, 0.8175195 , 0.85873604, 0.85469913, 0.8215766 ],
       [0.10262108, 0.10378427, 0.10396118, 0.11493546, 0.11241464,
        0.11194698, 0.11147197, 0.1091447 , 0.0981717 , 0.10589054,
        0.10181652, 0.09742663, 0.10040236, 0.09703647, 0.10117863,
        0.09760338, 0.10590921, 0.11065142, 0.10903537, 0.12229924,
        0.12367566, 0.13934848, 0.133329  , 0.1

In [7]:
from jax import random
import numpy as np
from jax import numpy as jnp
from jax import jit

key = random.PRNGKey(0)

def selu(x, alpha=1.67, lmbda=1.05):
    return lmbda * jnp.where(x > 0, x, alpha * jnp.exp(x) - alpha)

selu_jit = jit(selu)
x = random.normal(key, (1000000,))
%timeit selu(x).block_until_ready()
%timeit selu_jit(x).block_until_ready()

627 μs ± 142 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)
71 μs ± 3.98 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [9]:
jax.devices()
x = jnp.arange(10)